In [18]:
!pip install kagglehub torch -qU

In [19]:
import kagglehub
import pandas as pd
import os
# Download the dataset files

path = kagglehub.dataset_download("mexwell/poem-dataset")
print("Dataset downloaded to:", path)

data = pd.read_csv(os.path.join(path, os.listdir(path)[0]))

Using Colab cache for faster access to the 'poem-dataset' dataset.
Dataset downloaded to: /kaggle/input/poem-dataset


In [20]:
poems = data['poem content']

## Char level

In [21]:
cleaned = ""
for poem in poems:
  cleaned += "<START>\n"
  cleaned += poem.strip().lower()
  cleaned += "\n<END>\n\n"

In [22]:
import re
def preprocess_text(txt):
  txt = txt.replace("\r\n", "\n").replace("\r", "\n")
  txt = re.sub(r'[^a-z\n\s\',.:;!<>]', '', txt)
  txt = re.sub(r' +', ' ', txt)       # multiple spaces → one
  txt = re.sub(r'\n{3,}', '\n\n', txt)
  return txt
cleaned = preprocess_text(cleaned)

In [23]:
chars = sorted(set(cleaned))
print(chars)

['\n', ' ', '!', "'", ',', '.', ':', ';', '<', '>', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [24]:
print(set(cleaned))

{'t', 'u', ';', 'v', 'l', 'b', 'c', ',', 'y', 'm', ' ', 'j', 'o', 'e', 'h', 'd', '>', "'", 'r', 'z', 's', 'p', '\n', 'i', 'q', 'k', '.', ':', '!', 'x', 'w', 'n', 'g', 'a', 'f', '<'}


In [25]:
char2id = {c:i for i,c in enumerate(chars)}
id2char = {i:c for c,i in char2id.items()}

In [26]:
SEQ_LEN = 128
VOCAB_SIZE=len(chars)

In [34]:
X,y = [],[]
for i in range(len(cleaned) - SEQ_LEN):
    X.append([char2id[c] for c in cleaned[i:i+SEQ_LEN]])
    y.append([char2id[c] for c in cleaned[i+1:i+SEQ_LEN+1]])


In [35]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X = torch.tensor(X,dtype=torch.long).to(device)
y = torch.tensor(y,dtype=torch.long).to(device)

In [51]:
import torch.nn as nn

class LSTM(nn.Module):
  def __init__(self, VOCAB_SIZE, embedding_dim=64,hidden=256,layers=3,dropout=0):
    super().__init__()
    self.embed = nn.Embedding(VOCAB_SIZE, embedding_dim)
    self.lstm = nn.LSTM(embedding_dim, hidden, layers,batch_first=True)
    self.linear = nn.Linear(hidden, VOCAB_SIZE)

  def forward(self, x, hidden=None):
    embedded = self.embed(x)
    x, hidden = self.lstm(embedded,hidden)
    return self.linear(x), hidden

In [52]:
model = LSTM(VOCAB_SIZE)
model.to(device)

LSTM(
  (embed): Embedding(36, 64)
  (lstm): LSTM(64, 256, num_layers=3, batch_first=True)
  (linear): Linear(in_features=256, out_features=36, bias=True)
)

In [53]:
from torch.optim import Adam
from torch.nn import CrossEntropyLoss

optimizer = Adam(model.parameters(), lr=3e-3)
loss_fn = CrossEntropyLoss()

In [56]:
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 64
EPOCHS = 10

dataset = TensorDataset(X,y)
loader = DataLoader(dataset, batch_size=BATCH_SIZE,shuffle=True)

In [57]:

from tqdm import tqdm
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    loop = tqdm(loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS}", leave=True)
    for xb, yb in loop:
        optimizer.zero_grad()
        logits, _ = model(xb)
        loss = loss_fn(logits.view(-1, VOCAB_SIZE), yb.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1:02d} | Avg Loss: {avg_loss:.4f}")

Epoch 01/10: 100%|██████████| 5213/5213 [02:57<00:00, 29.38it/s, loss=0.3309]


Epoch 01 | Avg Loss: 0.8175


Epoch 02/10: 100%|██████████| 5213/5213 [03:04<00:00, 28.25it/s, loss=0.2559]


Epoch 02 | Avg Loss: 0.2771


Epoch 03/10: 100%|██████████| 5213/5213 [03:05<00:00, 28.10it/s, loss=0.2250]


...

Epoch 08 | Avg Loss: 0.2067


Epoch 09/10: 100%|██████████| 5213/5213 [03:14<00:00, 26.85it/s, loss=0.2138]


Epoch 09 | Avg Loss: 0.2040


Epoch 10/10: 100%|██████████| 5213/5213 [03:13<00:00, 26.87it/s, loss=0.2006]

Epoch 10 | Avg Loss: 0.2020


In [61]:
def generate(seed, length=500, temperature=1.0):
    model.eval()
    chars_out = list(seed)
    hidden = None
    seq = [char2id[c] for c in seed]

    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor([seq[-SEQ_LEN:]], dtype=torch.long).to(device)
            logits, hidden = model(x, hidden)

            # Take only the LAST timestep → (vocab,)
            last_logit = logits[0, -1, :]
            probs = torch.softmax(last_logit / temperature, dim=-1)
            next_idx = torch.multinomial(probs, 1).item()

            seq.append(next_idx)
            chars_out.append(id2char[next_idx])

    return "".join(chars_out)

print(generate("early to bed  ", length=300, temperature=0.8))

early to bed  ustin's mother.
never knew till now
either whom to love or how;
but be glad, as soon with me,
when you know that this is she
of whose be one her eyes,
her forehead where the golden light of evening spread,
the curve of heavy of thy dreams are all,
and pain cellelenting to your place.
but the goatgod
